
<img src=https://www.factset.com/hubfs/Assets/images/factset-logo.svg width="300" align="left">


# FactSet Supply Chain API - Getting Started
This notebook demonstrates basic features of the FactSet Supply Chain API by walking through the following steps:

1. Import Python packages 

2. Enter your Username and API Key for authorization

3. For each Supply Chain API endpoint, create request objects and display the results in a Pandas DataFrame

Additional Materials:  

* [FactSet Developer Portal](https://developer.factset.com/api-catalog/factset-supply-chain-api)


## 1. Import the required packages

In [ ]:
import requests
import json
import time
import pandas as pd
from requests.packages.urllib3.exceptions import InsecureRequestWarning
requests.packages.urllib3.disable_warnings(InsecureRequestWarning)
from pandas import json_normalize

import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
USERNAME = os.getenv("USERNAME")
APIKEY = os.getenv("APIKEY")

SUPPLY_CHAIN_URL = 'https://api.factset.com/content/factset-supply-chain/v1/relationships'

## 2. Create a connection object

Enter your credentials for 'Username' and 'API Key' variables below.

To generate an API key, visit  **[Manage API Keys](https://developer.factset.com/factset/api-key-listing)**. Click [here](https://developer.factset.com/authentication) for more details on Authentication.

Place your Username and API key in a `.env` file:


In [ ]:
authorization = (USERNAME, APIKEY)
headers = {'Accept': 'application/json', 'Content-Type': 'application/json'}

## Helper functions

In [ ]:
def reorder_columns(df, leading_cols=("requestId", "companyName")):
    """Move leading_cols to the front of the DataFrame, keeping the rest in original order."""
    front = [c for c in leading_cols if c in df.columns]
    rest = [c for c in df.columns if c not in front]
    return df[front + rest]


def fetch_relationships(ids, relationship_type, company_type="PUBLIC_COMPANIES_ONLY", direction="ALL"):
    """Fetch supply chain relationships and return a cleaned DataFrame."""
    request_body = {
        "data": {
            "ids": ids if isinstance(ids, list) else [ids],
            "relationshipType": relationship_type,
            "companyType": company_type,
            "relationshipDirection": direction
        }
    }

    response = requests.post(
        url=SUPPLY_CHAIN_URL,
        data=json.dumps(request_body),
        auth=authorization,
        headers=headers,
        verify=False
    )
    print(f"{relationship_type} - HTTP Status: {response.status_code}")

    if response.status_code != 200:
        print(f"Error: {response.text[:300]}")
        return pd.DataFrame()

    df = json_normalize(response.json()['data'])
    df = reorder_columns(df, leading_cols=("requestId", "companyName"))
    print(f"Records: {len(df)}, Columns: {len(df.columns)}")
    time.sleep(0.15)
    return df

# 3.0 FactSet Supply Chain API Endpoint Details

Section 3 includes detail for each FactSet Supply Chain API endpoint.  

The notebook creates a requests object and displays a DataFrame for each of the following FactSet Supply Chain API endpoints:

1. **Relationships**- [/relationships](#relationships)

For additional details regarding each endpoint's request parameters or response models, visit the [FactSet Supply Chain](https://developer.factset.com/api-catalog/factset-supply-chain-api) specification page.


<a id='Relationships'></a>
## 3.1 Relationships
Gets the count and percentage of overlapping products, along with the entity ID and associated company names for categories such as Supplier, Competitor, Customer, and Partner, based on the requested identifier(s).

### 3.1a `/relationships` - Fetch Customers for Novo Nordisk

In [ ]:
relationships_df = fetch_relationships("NVO-USA", "CUSTOMERS")
display(relationships_df)